In [1]:
import pandas as pd

In [2]:
# === Полный шаблон промпта с настройками ===
import requests
from copy import deepcopy

API_URL = "http://localhost:8080/v1/chat/completions"

SYSTEM_PROMPT_TEMPLATE = """Ты — NER-модель, специализирующаяся на извлечении геополитических субъектов из текста.\n\n🔹 Твоя задача — выделить все сущности, относящиеся к геополитике:\n• страны (Россия, США, Канада, Китай)\n• регионы и территории (Крым, Донбасс, Чечня, Гонконг, Тайвань)\n• города и административные центры (Москва, Киев, Вашингтон, Берлин)\n• международные организации и альянсы (НАТО, ЕС, ООН, БРИКС, ОПЕК, ШОС, G7)\n\nВсе остальное не включай ни в коем случае.\n\nПримеры:\nПрезидент США посетил Канаду → [\"США\", \"Канада\"]\nЕС ввёл санкции против России → [\"ЕС\", \"Россия\"]\nНАТО провело военные учения в Польше → [\"НАТО\", \"Польша\"]\nВ Крыму прошли выборы → [\"Крым\"]\nООН призвала Израиль и Палестину к миру → [\"ООН\", \"Израиль\", \"Палестина\"]\nБРИКС договорился о создании банка → [\"БРИКС\"]\nБайден встретился с Путиным → [\"США\", \"Россия\"]\n\nТеперь обработай:\n%s"
"""

# === 🔹 Base API request template ===
PROMPT_TEMPLATE = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT_TEMPLATE},
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "%s"},
    ],
    "max_tokens": 500,
    "temperature": 0.1,
    "top_p": 0.9
}


def get_prompt(text: str) -> str:
    prompt = deepcopy(PROMPT_TEMPLATE)
    prompt["messages"][0]["content"] = prompt["messages"][0]["content"] % text
    return prompt


def extract_countries_llm(text: str, prompt_template: dict = PROMPT_TEMPLATE, api_url: str = API_URL) -> list:
    """
    Отправляет текст в LLM, используя шаблон PROMPT_TEMPLATE.
    """
    prompt = get_prompt(text)

    # Отправляем запрос
    response = requests.post(api_url, json=prompt, timeout=60)
    response.raise_for_status()
    data = response.json()

    # Извлекаем контент
    content = data["choices"][0]["message"]["content"]

    # Пробуем распарсить JSON
    try:
        parsed = json.loads(content)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError as e:
        print(e)
        pass

    # fallback: простая очистка
    print(f'Не удалось распарсить {f}')
    return []



In [3]:
example = "В аэропорту имени Чан Кайши при взлёте разбился Boeing 747 компании Singapore Airlines, погибли 83 из 179 человек на борту."
countries = extract_countries_llm(example)
print("🌍 Найденные страны:", countries)

KeyboardInterrupt: 

In [79]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')
df = df.sample(n=100, random_state=42).reset_index(drop=True)
df

,date_start,date_end,event
0,2007-06-29,NaN,Опубликована окончательная версия GPLv3.
1,2021-01-01,NaN,Снижение экспортных пошлин на нефть и нефтепро...
2,2015-01-22,NaN,Президент йемена абд раббо мансур хади подал в...
3,2018-05-20,NaN,В должность президента черногории вступил мило...
4,2019-11-01,NaN,"Верховный суд России удовлетворил иск минюста,..."
...,...,...,...
95,2005-04-21,NaN,В Ираке сбит транспортный вертолёт Ми-8. Все д...
96,2010-10-07,NaN,Роскосмос запустил с космодрома байконур союз ...
97,2024-09-27,NaN,Взрыв на автозаправке в Дагестане. Погибли 13 ...
98,2000-03-29,NaN,Вторая чеченская война: гибель подразделения п...


In [84]:
import os
import pandas as pd
from tqdm import tqdm

# === Путь для чекпоинта ===
SAVE_DIR = "../data/events/3_countries"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, "countries_progress.csv")

# === Загружаем прогресс, если он есть ===
if os.path.exists(SAVE_PATH):
    df = pd.read_csv(SAVE_PATH)
    print(f"🔄 Найден сохранённый прогресс: {SAVE_PATH}")
else:
    df = pd.read_csv("../data/events/2_struct/2000-2025.csv")

    if "countries" not in df.columns:
        df["countries"] = None
    print("🆕 Загружен новый датасет событий.")

# === Параметры ===
SAVE_EVERY = 10  # каждые 10 строк сохраняем чекпоинт

# === Индексы строк, где ещё нет результатов ===
pending = df["countries"].isna().to_numpy().nonzero()[0]

print(f"📊 Всего записей: {len(df)}, ещё не обработано: {len(pending)}")

# === Основной цикл обработки ===
for j, i in enumerate(tqdm(pending, total=len(pending), desc="Извлечение стран")):
    try:
        current_value = df.at[i, "countries"]
        if pd.notna(current_value):
            continue
        result = extract_countries_llm(df.at[i, "event"])
        df.at[i, "countries"] = result
    except Exception as e:
        df.at[i, "countries"] = f"ERROR: {e}"

    # === Сохраняем чекпоинт каждые N за`писей ===
    if (j + 1) % SAVE_EVERY == 0 or (j + 1) == len(pending):
        df.to_csv(SAVE_PATH, index=False, encoding="utf-8")
        print(f"💾 Промежуточное сохранение ({j + 1}/{len(pending)}): {SAVE_PATH}")

# === Финальное сохранение ===
df.to_csv(SAVE_PATH, index=False, encoding="utf-8")
print("🎉 Обработка завершена и сохранена:", SAVE_PATH)

🆕 Загружен новый датасет событий.
📊 Всего записей: 5644, ещё не обработано: 5644


Извлечение стран:   0%|          | 1/5644 [00:02<4:19:45,  2.76s/it]

['Белоруссия']


Извлечение стран:   0%|          | 1/5644 [00:05<8:24:43,  5.37s/it]


KeyboardInterrupt: 

In [162]:
import ast
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import json
import time

start_time = time.time()

# === 🔹 Настройки ===
TEST_PATH = "../data/events/3_countries/countries_regress.csv"
REPORTS_DIR = "./reports"
MODEL_NAME = "Qwen3-8B-Q4_K_M."

# === 🔹 Загрузка данных ===
df = pd.read_csv(TEST_PATH)
assert "event" in df.columns and "countries" in df.columns, "В файле должны быть колонки 'event' и 'countries'"


def parse_ref(s):
    """Безопасно преобразует строку вида "['Россия','Беларусь']" в список."""
    if pd.isna(s):
        return []
    if isinstance(s, list):
        return s
    try:
        parsed = ast.literal_eval(str(s))
        if isinstance(parsed, list):
            # очищаем элементы от лишних кавычек и пробелов
            return [x.strip().strip("'").strip('"') for x in parsed]
    except Exception:
        pass
    # fallback: грубое разделение по запятым
    return [x.strip().strip("'").strip('"') for x in str(s).strip("[]").split(",") if x.strip()]


df["expected"] = df["countries"].apply(parse_ref)

# === 🔹 Основной цикл проверки ===
results = []
mismatches = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="🔍 Проверка событий"):
    event = row["event"]
    expected = set(row["expected"])

    try:
        predicted = set(extract_countries_llm(event))
    except Exception as e:
        predicted = set()
        print(f"⚠️ Ошибка на строке {idx}: {e}")

    # Проверяем совпадение (без учёта порядка)
    correct = expected == predicted

    results.append({
        "event": event,
        "expected": list(expected),
        "predicted": list(predicted),
        "correct": correct
    })

    if not correct:
        print("\n❌ Несовпадение:")
        print(f"Событие: {event}")
        print(f"Эталон:   {list(expected)}")
        print(f"Модель:   {list(predicted)}")

# === 🔹 Формирование отчёта ===
report_df = pd.DataFrame(results)
total = len(report_df)
correct = report_df["correct"].sum()
accuracy = correct / total * 100

# === 🔹 Сохранение отчёта ===
now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
report_path = f"{REPORTS_DIR}/llm_test_report_{now}.txt"

# создаём папку, если нет
import os

os.makedirs(REPORTS_DIR, exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"LLM Benchmark Report — {now}\n")
    f.write(f"Модель: {MODEL_NAME}\n")
    f.write(f'Промпт: {get_prompt("test-prompt")}\n')
    f.write(f"Файл теста: {TEST_PATH}\n\n")
    f.write(f"✅ Совпало: {correct} / {total} ({accuracy:.2f}%)\n")
    f.write(f'Время: {time.time() - start_time}\n\n')
    f.write("🔹 Ошибки:\n")
    for r in results:
        if not r["correct"]:
            f.write(f"\nСобытие: {r['event']}\n")
            f.write(f"Эталон: {r['expected']}\n")
            f.write(f"Модель: {r['predicted']}\n")

print(f"\n📊 Отчёт сохранён: {report_path}")
print(f"Совпало {correct} из {total} ({accuracy:.2f}%)")

🔍 Проверка событий:   3%|▎         | 3/100 [00:07<04:05,  2.53s/it]


❌ Несовпадение:
Событие: Президент йемена абд раббо мансур хади подал в отставку.
Эталон:   ['Йемен']
Модель:   []


🔍 Проверка событий:  13%|█▎        | 13/100 [00:33<03:40,  2.53s/it]


❌ Несовпадение:
Событие: В западном иране произошло землетрясение магнитудой 6,5, погибли более 260 человек.
Эталон:   ['Иран']
Модель:   []


🔍 Проверка событий:  16%|█▌        | 16/100 [00:41<03:38,  2.60s/it]

...


❌ Несовпадение:
Событие: Роскосмос запустил с космодрома байконур союз «союз тма-01м» с модернизированной системой управления. экипаж старта — александр калери, олег скрипочка и скотт келли (сша).
Эталон:   ['Россия', 'США', 'Байконур']
Модель:   ['Байконур', 'США', 'Олег Скрипочка', 'Александр Калери', 'Скотт Келли', 'Союз', 'Роскосмос', 'Союз ТМА-01М']


🔍 Проверка событий:  99%|█████████▉| 99/100 [04:24<00:02,  2.90s/it]


❌ Несовпадение:
Событие: Вторая чеченская война: гибель подразделения пермского омона у селения джани-ведено. погибло более 40 человек.
Эталон:   ['Пермский край', 'Чечня']
Модель:   ['Россия', 'Чечня']


🔍 Проверка событий: 100%|██████████| 100/100 [04:27<00:00,  2.67s/it]


❌ Несовпадение:
Событие: Российский оппозиционный политик алексей навальный госпитализирован в омскую больницу с отравлением, откуда позже был перевезён в германию.
Эталон:   ['Россия', 'Германия', 'Омск']
Модель:   ['Россия', 'Германия']

📊 Отчёт сохранён: ./reports/llm_test_report_2025-10-17_20-56-09.txt
Совпало 61 из 100 (61.00%)
